# IID Experiments — Single-class & Multi-class (Pavia-U)

**No PCA / No AE** — every detector receives raw 103-D bands.  
Score nets (DSM, LRao) carry a **frozen ZCA whitening first layer** (fit on training background; relative eigenvalue floor).  
Two sweeps per run:
- **vs n** at fixed ρ: AUC, partial-AUC (Pfa<0.1), Pd@Pfa
- **vs ρ** at fixed n: AUC, Pd@Pfa

Detectors — single: **AMF, DSM, LRao** | multi: **AMF, GMM-Levin, DSM, LRao**  
Outputs per run: `roc_at_nmax.pdf`, `roc_at_nfixed.pdf`, `auc_vs_n.pdf`, `pauc_vs_n.pdf`, `pd_at_fa_vs_n.pdf`, `auc_vs_rho.pdf`, `pdet_at_pfa_vs_rho.pdf`

**Runtime**: single-class ≈15–30 min on T4 GPU (full config); multi-class ≈20–40 min.  
Use `QUICK = True` in Cell 4 to smoke-test in <5 min.

In [ ]:
# ── Cell 1: Clone / pull repo + install deps ──────────────────────────────
import os, sys
REPO_URL = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL    = '/content/proj'
BRANCH   = 'iid-unified-experiment'

if os.path.exists(os.path.join(LOCAL, '.git')):
    !git -C {LOCAL} fetch --all -q
    !git -C {LOCAL} checkout {BRANCH} -q
    !git -C {LOCAL} pull origin {BRANCH} -q
else:
    !git clone -b {BRANCH} --depth 1 {REPO_URL} {LOCAL}

!pip install -q scikit-learn scipy tqdm matplotlib pyyaml einops

if LOCAL not in sys.path: sys.path.insert(0, LOCAL)
os.chdir(LOCAL)
print('CWD:', os.getcwd())
!git -C {LOCAL} log --oneline -1

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — running on CPU. Enable GPU in Runtime > Change runtime type.')
print(f'PyTorch {torch.__version__}  device={DEVICE}')

In [ ]:
# ── Cell 3: Mount Drive + link pavia-u.mat ────────────────────────────────
import os
RESULTS = '/content/results'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/final_paper/iid_results'
    os.makedirs('/content/proj/data', exist_ok=True)
    DST = '/content/proj/data/pavia-u.mat'
    if not os.path.exists(DST):
        for src in [
            '/content/drive/MyDrive/final_paper/pavia-u.mat',
            '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat',
        ]:
            if os.path.exists(src):
                os.symlink(src, DST)
                print('Linked dataset from', src)
                break
        else:
            print('WARNING: pavia-u.mat not found on Drive.')
    else:
        print('Dataset already linked.')
except Exception as e:
    print('Drive not mounted:', repr(e))

os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), \
    'pavia-u.mat missing! Mount Drive and re-run this cell.'
print('RESULTS dir:', RESULTS)

In [ ]:
# ── Cell 4: Config overrides ───────────────────────────────────────────────
# QUICK=True  → fast smoke-test  (~3 min, 2 n-values, 200 epochs)
# QUICK=False → full paper run   (~30 min single, ~40 min multi on T4)
QUICK = False

QUICK_OVERRIDES = dict(
    n_train_list    = [100, 500],
    rho_list        = [0.003, 0.01, 0.1],
    n_fixed_for_rho = 100,
    dsm_epochs      = 200,
    lrao_epochs     = 30,
    test_size       = 300,
    device          = DEVICE,
)
FULL_OVERRIDES = dict(device=DEVICE)
OVERRIDES = QUICK_OVERRIDES if QUICK else FULL_OVERRIDES
print('Mode:', 'QUICK smoke-test' if QUICK else 'FULL paper run')
print('Overrides:', OVERRIDES)

---
## Single-class IID  (bkg = asphalt cls1,  tgt = trees cls3)

In [ ]:
# ── Cell S1: Run single-class IID ─────────────────────────────────────────
import yaml, os
from iid_core import run_iid

with open('experiments/iid_single/config.yaml') as f:
    cfg_single = yaml.safe_load(f)

cfg_single.update(OVERRIDES)
cfg_single['results_dir'] = os.path.join(RESULTS, 'iid_single')
os.makedirs(cfg_single['results_dir'], exist_ok=True)

print('=== IID SINGLE-CLASS ===')
print(f"bkg_cls    : {cfg_single.get('bkg_cls')}   target_cls: {cfg_single.get('target_cls')}")
print(f"n_list     : {cfg_single.get('n_train_list')}")
print(f"rho_list   : {cfg_single.get('rho_list', '[default]')}")
print(f"dsm_epochs : {cfg_single.get('dsm_epochs')}")
print(f"device     : {cfg_single.get('device')}")
print()

run_dir_single, metrics_single = run_iid(cfg_single, mode='single')
print('\nSingle run dir:', run_dir_single)

In [ ]:
# ── Cell S2: Display single-class figures ─────────────────────────────────
import glob, os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir_s = os.path.join(run_dir_single, 'figures')

FIGS = [
    ('roc_at_nmax.pdf',        'ROC — largest n'),
    ('roc_at_nfixed.pdf',      'ROC — fixed n, sweep ρ'),
    ('auc_vs_n.pdf',           'AUC vs n'),
    ('pauc_vs_n.pdf',          'Partial-AUC (Pfa<0.1) vs n'),
    ('pd_at_fa_vs_n.pdf',      'Pd @ Pfa=0.1 vs n'),
    ('auc_vs_rho.pdf',         'AUC vs ρ'),
    ('pdet_at_pfa_vs_rho.pdf', 'Pd @ Pfa vs ρ'),
]

def show_pdf(pdf_path, title):
    png = pdf_path.replace('.pdf', '.png')
    os.system(f'convert -density 150 "{pdf_path}" "{png}" 2>/dev/null || '
              f'pdftoppm -r 150 -png "{pdf_path}" "{png[:-4]}" 2>/dev/null')
    img = png if os.path.exists(png) else (png[:-4]+'-1.png' if os.path.exists(png[:-4]+'-1.png') else None)
    if img:
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.imshow(mpimg.imread(img)); ax.axis('off')
        ax.set_title(title, fontsize=10); plt.tight_layout(); plt.show()
    else:
        print(f'  {title}: PDF saved at {pdf_path}')

for fname, title in FIGS:
    p = os.path.join(fig_dir_s, fname)
    if os.path.exists(p): show_pdf(p, f'[single] {title}')
    else: print(f'[single] {title}: not found')

for lp in glob.glob(os.path.join(fig_dir_s, 'loss_curves_*.png')):
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.imshow(mpimg.imread(lp)); ax.axis('off')
    ax.set_title('[single] Training loss curves', fontsize=10)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Cell S3: Single-class numeric summary ─────────────────────────────────
import json, pandas as pd

with open(os.path.join(run_dir_single, 'metrics.json')) as f:
    m = json.load(f)

dets = list(m['vs_n'].keys())
print('=== vs-n  AUC (single-class) ===')
print(pd.DataFrame({d: m['vs_n'][d]['auc'] for d in dets},
                   index=m['n_list']).rename_axis('n').round(3).to_string())

print('\n=== vs-ρ  AUC (single-class) ===')
print(pd.DataFrame({d: m['vs_rho'][d]['auc'] for d in dets},
                   index=m['rho_list']).rename_axis('rho').round(3).to_string())

if 'roc_at_nmax' in m:
    print(f"\n=== ROC AUC at n={m['n_list'][-1]} ===")
    for d, v in m['roc_at_nmax'].items():
        print(f'  {d:12s}: {v["auc"]:.4f}')

---
## Multi-class IID  (tgt = trees cls3,  bkg = all other classes)

In [ ]:
# ── Cell M1: Run multi-class IID ──────────────────────────────────────────
import yaml, os
from iid_core import run_iid

with open('experiments/iid_multi/config.yaml') as f:
    cfg_multi = yaml.safe_load(f)

cfg_multi.update(OVERRIDES)
cfg_multi['results_dir'] = os.path.join(RESULTS, 'iid_multi')
os.makedirs(cfg_multi['results_dir'], exist_ok=True)

print('=== IID MULTI-CLASS ===')
print(f"target_cls : {cfg_multi.get('target_cls')}  (bkg = all other classes)")
print(f"n_list     : {cfg_multi.get('n_train_list')}")
print(f"rho_list   : {cfg_multi.get('rho_list', '[default]')}")
print(f"gmm_K      : {cfg_multi.get('gmm_K')}")
print(f"device     : {cfg_multi.get('device')}")
print()

run_dir_multi, metrics_multi = run_iid(cfg_multi, mode='multi')
print('\nMulti run dir:', run_dir_multi)

In [ ]:
# ── Cell M2: Display multi-class figures ──────────────────────────────────
import glob, os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir_m = os.path.join(run_dir_multi, 'figures')
FIGS = [
    ('roc_at_nmax.pdf',        'ROC — largest n'),
    ('roc_at_nfixed.pdf',      'ROC — fixed n, sweep ρ'),
    ('auc_vs_n.pdf',           'AUC vs n'),
    ('pauc_vs_n.pdf',          'Partial-AUC (Pfa<0.1) vs n'),
    ('pd_at_fa_vs_n.pdf',      'Pd @ Pfa=0.1 vs n'),
    ('auc_vs_rho.pdf',         'AUC vs ρ'),
    ('pdet_at_pfa_vs_rho.pdf', 'Pd @ Pfa vs ρ'),
]

for fname, title in FIGS:
    p = os.path.join(fig_dir_m, fname)
    if os.path.exists(p): show_pdf(p, f'[multi] {title}')
    else: print(f'[multi] {title}: not found')

for lp in glob.glob(os.path.join(fig_dir_m, 'loss_curves_*.png')):
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.imshow(mpimg.imread(lp)); ax.axis('off')
    ax.set_title('[multi] Training loss curves', fontsize=10)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Cell M3: Multi-class numeric summary ──────────────────────────────────
import json, pandas as pd

with open(os.path.join(run_dir_multi, 'metrics.json')) as f:
    mm = json.load(f)

dets_m = list(mm['vs_n'].keys())
print('=== vs-n  AUC (multi-class) ===')
print(pd.DataFrame({d: mm['vs_n'][d]['auc'] for d in dets_m},
                   index=mm['n_list']).rename_axis('n').round(3).to_string())

print('\n=== vs-ρ  AUC (multi-class) ===')
print(pd.DataFrame({d: mm['vs_rho'][d]['auc'] for d in dets_m},
                   index=mm['rho_list']).rename_axis('rho').round(3).to_string())

if 'roc_at_nmax' in mm:
    print(f"\n=== ROC AUC at n={mm['n_list'][-1]} (multi) ===")
    for d, v in mm['roc_at_nmax'].items():
        print(f'  {d:12s}: {v["auc"]:.4f}')

---
## Single vs Multi — Side-by-side comparison

In [ ]:
# ── Cell C1: AUC vs n — single vs multi ───────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mticker

COLORS  = {'AMF':'#1f77b4','GMM-Levin':'#9467bd','DSM':'#d62728','LRao':'#2ca02c'}
MARKERS = {'AMF':'o','GMM-Levin':'^','DSM':'D','LRao':'D'}
LW      = {'AMF':1.4,'GMM-Levin':1.4,'DSM':2.2,'LRao':2.2}

def _auc_vs_n(ax, n_list, met, title):
    x = np.array(n_list, dtype=float)
    for d in met['vs_n']:
        ax.plot(x, met['vs_n'][d]['auc'], color=COLORS.get(d,'#888'),
                lw=LW.get(d,1.4), marker=MARKERS.get(d,'o'), label=d)
    ax.set_xscale('log')
    ax.set_xticks(x)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:g}'))
    ax.xaxis.set_minor_locator(mticker.NullLocator())
    ax.tick_params(axis='x', labelsize=8, rotation=45)
    ax.set_xlabel('n'); ax.set_ylabel('AUC')
    ax.set_title(title, fontsize=9); ax.grid(alpha=0.25)
    ax.legend(fontsize=7, loc='lower right')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
_auc_vs_n(ax1, metrics_single['n_list'], metrics_single, 'AUC vs n  [single-class]')
_auc_vs_n(ax2, metrics_multi['n_list'],  metrics_multi,  'AUC vs n  [multi-class]')
fig.suptitle('AUC vs training size', fontsize=10)
fig.tight_layout()
plt.savefig(os.path.join(RESULTS, 'compare_auc_vs_n.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell C2: AUC vs ρ — single vs multi ───────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mticker

def _auc_vs_rho(ax, rho_list, met, title):
    x = np.array(rho_list, dtype=float)
    for d in met['vs_rho']:
        ax.plot(x, met['vs_rho'][d]['auc'], color=COLORS.get(d,'#888'),
                lw=LW.get(d,1.4), marker=MARKERS.get(d,'o'), label=d)
    ax.set_xscale('log')
    ax.set_xticks(x)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:g}'))
    ax.xaxis.set_minor_locator(mticker.NullLocator())
    ax.tick_params(axis='x', labelsize=8, rotation=45)
    ax.set_xlabel(r'$\rho$'); ax.set_ylabel('AUC')
    ax.set_title(title, fontsize=9); ax.grid(alpha=0.25)
    ax.legend(fontsize=7, loc='lower right')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
_auc_vs_rho(ax1, metrics_single['rho_list'], metrics_single,
            f'AUC vs ρ  [single, n={metrics_single["n_fixed"]}]')
_auc_vs_rho(ax2, metrics_multi['rho_list'],  metrics_multi,
            f'AUC vs ρ  [multi, n={metrics_multi["n_fixed"]}]')
fig.suptitle(r'AUC vs DSM noise level $\rho$', fontsize=10)
fig.tight_layout()
plt.savefig(os.path.join(RESULTS, 'compare_auc_vs_rho.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell D1: Download all figures as a zip ────────────────────────────────
import zipfile, glob, os
from google.colab import files

ZIP_PATH = '/content/iid_figures.zip'
to_zip = []
for rd in [run_dir_single, run_dir_multi]:
    for ext in ['*.pdf', '*.png']:
        to_zip.extend(glob.glob(os.path.join(rd, 'figures', ext)))
for cmp in ['compare_auc_vs_n.pdf', 'compare_auc_vs_rho.pdf']:
    p = os.path.join(RESULTS, cmp)
    if os.path.exists(p): to_zip.append(p)

base = os.path.dirname(run_dir_single.rstrip('/'))
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in to_zip:
        z.write(p, os.path.relpath(p, base))

print(f'Zipped {len(to_zip)} files → {ZIP_PATH}')
files.download(ZIP_PATH)